<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Multi-Target/MultiTarget_Statistical_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import gc
import warnings

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.api import VAR

warnings.filterwarnings("ignore")

In [2]:
multi_target_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Multi Target/cleaned_multi_target_dataset (1).csv")

print("Dataset shape:", multi_target_df.shape)
multi_target_df.head()

Dataset shape: (130003, 18)


,visibility_km,last_updated_epoch,air_quality_Ozone,latitude,air_quality_PM10,condition_text,air_quality_Sulphur_dioxide,uv_index,air_quality_Nitrogen_dioxide,feels_like_celsius,precip_mm,cloud,pressure_mb,longitude,air_quality_Carbon_Monoxide,air_quality_PM2.5,temperature_celsius,humidity
0,16.0,1715849100,62.2,46.60,7.1,2,0.2,1.0,2.5,16.1,0.00,0,1012.0,-120.49,198.6,6.3,16.1,58
1,10.0,1715849100,23.3,14.10,25.3,32,1.4,1.0,3.7,25.3,0.28,37,1017.0,-87.22,377.2,19.0,23.0,78
2,10.0,1715849100,5.9,13.71,28.1,23,7.5,1.0,7.7,30.2,0.30,50,1010.0,-89.20,460.6,20.4,26.0,94
3,5.0,1715849100,0.4,14.62,178.1,19,19.3,1.0,35.0,20.0,0.09,100,1019.0,-90.53,2243.0,132.0,20.0,88
4,10.0,1715849100,34.0,17.25,32.1,30,0.2,1.0,0.3,29.6,0.00,94,1007.0,-88.77,307.1,7.7,26.0,89


In [3]:
targets = [
    "air_quality_PM2.5",
    "temperature_celsius",
    "humidity"
]

stat_df = multi_target_df[targets].copy()

print("Statistical dataset shape:", stat_df.shape)
stat_df.head()

Statistical dataset shape: (130003, 3)


,air_quality_PM2.5,temperature_celsius,humidity
0,6.3,16.1,58
1,19.0,23.0,78
2,20.4,26.0,94
3,132.0,20.0,88
4,7.7,26.0,89


In [4]:
split_index = int(0.8 * len(stat_df))

train_df = stat_df.iloc[:split_index].copy()
test_df = stat_df.iloc[split_index:].copy()

print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (104002, 3)
Test shape : (26001, 3)


In [5]:
def calculate_metrics_series(y_true, y_pred, model_name, dataset_type, target_name):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    acc = r2 * 100

    return pd.DataFrame([[
        dataset_type,
        model_name,
        target_name,
        mse,
        rmse,
        mae,
        r2,
        acc
    ]], columns=["Dataset", "Model", "Target", "MSE", "RMSE", "MAE", "R2", "Accuracy (%)"]).round(3)

In [6]:
arima_training_results = []
arima_testing_results = []

for target in targets:
    print(f"\nRunning ARIMA for {target}")

    train_series = train_df[target]
    test_series = test_df[target]

    # light ARIMA
    arima_model = ARIMA(
        train_series,
        order=(1, 1, 1)
    )
    arima_fit = arima_model.fit()

    # training prediction
    train_pred = arima_fit.predict(start=1, end=len(train_series) - 1)
    train_actual = train_series.iloc[1:]

    arima_training_results.append(
        calculate_metrics_series(
            train_actual,
            train_pred,
            model_name="ARIMA",
            dataset_type="Training",
            target_name=target
        )
    )

    # testing forecast
    test_pred = arima_fit.forecast(steps=len(test_series))

    arima_testing_results.append(
        calculate_metrics_series(
            test_series,
            test_pred,
            model_name="ARIMA",
            dataset_type="Testing",
            target_name=target
        )
    )

    del arima_model, arima_fit, train_pred, test_pred
    gc.collect()


Running ARIMA for air_quality_PM2.5

Running ARIMA for temperature_celsius

Running ARIMA for humidity


In [7]:
arima_training_table = pd.concat(arima_training_results, ignore_index=True)
arima_testing_table = pd.concat(arima_testing_results, ignore_index=True)

print("ARIMA TRAINING RESULTS")
display(arima_training_table)

print("ARIMA TESTING RESULTS")
display(arima_testing_table)

ARIMA TRAINING RESULTS


,Dataset,Model,Target,MSE,RMSE,MAE,R2,Accuracy (%)
0,Training,ARIMA,air_quality_PM2.5,1543.698,39.290,19.623,0.031,3.086
1,Training,ARIMA,temperature_celsius,64.727,8.045,6.242,0.174,17.351
2,Training,ARIMA,humidity,405.115,20.127,15.781,0.306,30.550


ARIMA TESTING RESULTS


,Dataset,Model,Target,MSE,RMSE,MAE,R2,Accuracy (%)
0,Testing,ARIMA,air_quality_PM2.5,610.825,24.715,13.870,-0.020,-2.025
1,Testing,ARIMA,temperature_celsius,155.339,12.464,9.273,-0.251,-25.074
2,Testing,ARIMA,humidity,741.477,27.230,24.380,-0.547,-54.673


In [8]:
sarima_training_results = []
sarima_testing_results = []

seasonal_period = 12   # keep as 12 unless you know a better seasonal cycle

for target in targets:
    print(f"\nRunning SARIMA for {target}")

    train_series = train_df[target]
    test_series = test_df[target]

    sarima_model = SARIMAX(
        train_series,
        order=(1, 1, 0),
        seasonal_order=(0, 1, 1, seasonal_period),
        trend="n",
        simple_differencing=True,
        enforce_stationarity=False,
        enforce_invertibility=False
    )

    sarima_fit = sarima_model.fit(
        disp=False,
        maxiter=30
    )

    # training prediction
    start_idx = seasonal_period
    train_pred = sarima_fit.predict(start=start_idx, end=len(train_series) - 1)
    train_actual = train_series.iloc[start_idx:]

    sarima_training_results.append(
        calculate_metrics_series(
            train_actual,
            train_pred,
            model_name="SARIMA",
            dataset_type="Training",
            target_name=target
        )
    )

    # testing forecast
    test_pred = sarima_fit.forecast(steps=len(test_series))

    sarima_testing_results.append(
        calculate_metrics_series(
            test_series,
            test_pred,
            model_name="SARIMA",
            dataset_type="Testing",
            target_name=target
        )
    )

    del sarima_model, sarima_fit, train_pred, test_pred
    gc.collect()


Running SARIMA for air_quality_PM2.5

Running SARIMA for temperature_celsius

Running SARIMA for humidity


In [9]:
sarima_training_table = pd.concat(sarima_training_results, ignore_index=True)
sarima_testing_table = pd.concat(sarima_testing_results, ignore_index=True)

print("SARIMA TRAINING RESULTS")
display(sarima_training_table)

print("SARIMA TESTING RESULTS")
display(sarima_testing_table)

SARIMA TRAINING RESULTS


,Dataset,Model,Target,MSE,RMSE,MAE,R2,Accuracy (%)
0,Training,SARIMA,air_quality_PM2.5,2975.325,54.547,30.603,-0.868,-86.788
1,Training,SARIMA,temperature_celsius,619.795,24.896,23.003,-6.913,-691.348
2,Training,SARIMA,humidity,4918.468,70.132,64.826,-7.432,-743.188


SARIMA TESTING RESULTS


,Dataset,Model,Target,MSE,RMSE,MAE,R2,Accuracy (%)
0,Testing,SARIMA,air_quality_PM2.5,1004.770,31.698,20.149,-0.678,-67.824
1,Testing,SARIMA,temperature_celsius,391.812,19.794,17.421,-2.155,-215.475
2,Testing,SARIMA,humidity,5899.506,76.808,73.620,-11.306,-1130.643


In [10]:
var_model = VAR(train_df)

lag_selection = var_model.select_order(maxlags=8)
print(lag_selection.summary())

best_lag = lag_selection.aic
print("Best lag by AIC:", best_lag)

# fallback if AIC returns None
if best_lag is None:
    best_lag = 2

var_fit = var_model.fit(best_lag)
print(var_fit.summary())

 VAR Order Selection (* highlights the minimums) 
      AIC         BIC         FPE         HQIC   
-------------------------------------------------
0       17.94       17.94   6.179e+07       17.94
1       17.68       17.68   4.776e+07       17.68
2       17.56       17.56   4.231e+07       17.56
3       17.49       17.50   3.955e+07       17.49
4       17.45       17.45   3.770e+07       17.45
5       17.41       17.41   3.640e+07       17.41
6       17.38       17.39   3.546e+07       17.39
7       17.37       17.37   3.492e+07       17.37
8      17.36*      17.36*  3.447e+07*      17.36*
-------------------------------------------------
Best lag by AIC: 8
  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 31, Mar, 2026
Time:                     06:37:35
--------------------------------------------------------------------
No. of Equations:         3.00000    BIC:                    17.3626
Nobs:             

In [11]:
var_train_pred = var_fit.fittedvalues.copy()
var_train_actual = train_df.iloc[best_lag:].copy()

print("VAR train actual shape:", var_train_actual.shape)
print("VAR train predicted shape:", var_train_pred.shape)

VAR train actual shape: (103994, 3)
VAR train predicted shape: (103994, 3)


In [12]:
var_training_results = []

for target in targets:
    var_training_results.append(
        calculate_metrics_series(
            var_train_actual[target],
            var_train_pred[target],
            model_name="VAR",
            dataset_type="Training",
            target_name=target
        )
    )

var_training_table = pd.concat(var_training_results, ignore_index=True)

print("VAR TRAINING RESULTS")
display(var_training_table)

VAR TRAINING RESULTS


,Dataset,Model,Target,MSE,RMSE,MAE,R2,Accuracy (%)
0,Training,VAR,air_quality_PM2.5,1536.118,39.193,19.494,0.036,3.561
1,Training,VAR,temperature_celsius,65.322,8.082,6.293,0.166,16.595
2,Training,VAR,humidity,405.002,20.125,15.900,0.306,30.570


In [13]:
forecast_input = train_df.values[-best_lag:]
forecast_steps = len(test_df)

var_forecast = var_fit.forecast(y=forecast_input, steps=forecast_steps)
var_test_pred = pd.DataFrame(var_forecast, columns=targets, index=test_df.index)

print("VAR test prediction shape:", var_test_pred.shape)
var_test_pred.head()

VAR test prediction shape: (26001, 3)


,air_quality_PM2.5,temperature_celsius,humidity
104002,25.200068,23.945245,52.873411
104003,24.424608,23.800498,55.943107
104004,23.256579,23.562491,57.146244
104005,23.842908,23.589666,60.339159
104006,23.533432,23.176920,60.535882


In [14]:
var_testing_results = []

for target in targets:
    var_testing_results.append(
        calculate_metrics_series(
            test_df[target],
            var_test_pred[target],
            model_name="VAR",
            dataset_type="Testing",
            target_name=target
        )
    )

var_testing_table = pd.concat(var_testing_results, ignore_index=True)

print("VAR TESTING RESULTS")
display(var_testing_table)

VAR TESTING RESULTS


,Dataset,Model,Target,MSE,RMSE,MAE,R2,Accuracy (%)
0,Testing,VAR,air_quality_PM2.5,627.852,25.057,17.362,-0.049,-4.869
1,Testing,VAR,temperature_celsius,164.014,12.807,9.416,-0.321,-32.059
2,Testing,VAR,humidity,560.635,23.678,20.612,-0.169,-16.949


In [15]:
training_table = pd.concat([
    arima_training_table,
    sarima_training_table,
    var_training_table
], ignore_index=True)

print("FINAL TRAINING TABLE")
display(training_table)

FINAL TRAINING TABLE


,Dataset,Model,Target,MSE,RMSE,MAE,R2,Accuracy (%)
0,Training,ARIMA,air_quality_PM2.5,1543.698,39.290,19.623,0.031,3.086
1,Training,ARIMA,temperature_celsius,64.727,8.045,6.242,0.174,17.351
2,Training,ARIMA,humidity,405.115,20.127,15.781,0.306,30.550
3,Training,SARIMA,air_quality_PM2.5,2975.325,54.547,30.603,-0.868,-86.788
4,Training,SARIMA,temperature_celsius,619.795,24.896,23.003,-6.913,-691.348
5,Training,SARIMA,humidity,4918.468,70.132,64.826,-7.432,-743.188
6,Training,VAR,air_quality_PM2.5,1536.118,39.193,19.494,0.036,3.561
7,Training,VAR,temperature_celsius,65.322,8.082,6.293,0.166,16.595
8,Training,VAR,humidity,405.002,20.125,15.900,0.306,30.570


In [16]:
testing_table = pd.concat([
    arima_testing_table,
    sarima_testing_table,
    var_testing_table
], ignore_index=True)

print("FINAL TESTING TABLE")
display(testing_table)

FINAL TESTING TABLE


,Dataset,Model,Target,MSE,RMSE,MAE,R2,Accuracy (%)
0,Testing,ARIMA,air_quality_PM2.5,610.825,24.715,13.870,-0.020,-2.025
1,Testing,ARIMA,temperature_celsius,155.339,12.464,9.273,-0.251,-25.074
2,Testing,ARIMA,humidity,741.477,27.230,24.380,-0.547,-54.673
3,Testing,SARIMA,air_quality_PM2.5,1004.770,31.698,20.149,-0.678,-67.824
4,Testing,SARIMA,temperature_celsius,391.812,19.794,17.421,-2.155,-215.475
5,Testing,SARIMA,humidity,5899.506,76.808,73.620,-11.306,-1130.643
6,Testing,VAR,air_quality_PM2.5,627.852,25.057,17.362,-0.049,-4.869
7,Testing,VAR,temperature_celsius,164.014,12.807,9.416,-0.321,-32.059
8,Testing,VAR,humidity,560.635,23.678,20.612,-0.169,-16.949


In [17]:
from google.colab import files

training_table.to_csv("statistical_models_training_results.csv", index=False)
testing_table.to_csv("statistical_models_testing_results.csv", index=False)

files.download("statistical_models_training_results.csv")
files.download("statistical_models_testing_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>